In [2]:
# pip install rapidfuzz

from rapidfuzz import fuzz, process

# --- 1. Basic similarity scores (0-100) ---
print(fuzz.ratio("hello world", "hello wrld"))          # simple similarity
print(fuzz.partial_ratio("hello", "say hello there"))     # best matching substring
print(fuzz.token_sort_ratio("new york mets", "mets new york"))  # ignores word order
print(fuzz.token_set_ratio("new york mets vs", "new york mets"))  # ignores duplicates/extra words

# --- 2. Find the best match in a list ---
choices = ["New York Mets", "New York Yankees", "Boston Red Sox", "Chicago Cubs"]

best_match = process.extractOne("ny mets", choices)
print(best_match)  # ('New York Mets', 90.0, 0) -> (match, score, index)

# --- 3. Get top N matches ---
top_matches = process.extract("new yrok", choices, limit=2)
print(top_matches)

# --- 4. Fuzzy matching with a score cutoff (filter out weak matches) ---
matches = process.extract("boston sox", choices, score_cutoff=60)
print(matches)

# --- 5. Deduplicating a list of similar strings ---
# NOTE: process.dedupe does not exist in current RapidFuzz — build it manually
def dedupe(strings, threshold=85):
    """Keep only one representative per group of near-duplicate strings."""
    kept = []
    for s in strings:
        match = process.extractOne(s, kept, scorer=fuzz.ratio, score_cutoff=threshold)
        if match is None:
            kept.append(s)
    return kept

data = ["apple inc", "apple incorporated", "Apple Inc.", "banana co", "Banana Co."]
deduped = dedupe(data, threshold=85)
print(deduped)

# --- 6. Matching against a dictionary (keys as choices, get back the key) ---
db = {"nyc": "New York City", "la": "Los Angeles", "sf": "San Francisco"}
match, score, key = process.extractOne("san fransisco", db)
print(match, score, key)

# --- 7. Full similarity matrix (all-pairs comparison, useful for clustering) ---
import numpy as np

matrix = process.cdist(data, data, scorer=fuzz.ratio)
print(np.round(matrix, 1))

95.23809523809523
100.0
100.0
100.0
('New York Mets', 60.00000000000001, 0)
[('New York Mets', 57.0, 0), ('New York Yankees', 56.25, 1)]
[('Boston Red Sox', 66.66666666666667, 2)]
['apple inc', 'apple incorporated', 'Apple Inc.', 'banana co', 'Banana Co.']
San Francisco 76.92307692307692 sf
[[100.   66.7  73.7  33.3  21.1]
 [ 66.7 100.   50.   29.6  21.4]
 [ 73.7  50.  100.   21.1  20. ]
 [ 33.3  29.6  21.1 100.   73.7]
 [ 21.1  21.4  20.   73.7 100. ]]
